# JSON 与数据转换

学习目标：能设计明确的 JSON 数据边界，正确使用转换回调，区分解析、校验与对象复制。

前置知识：对象、数组、字符串、数值精度、函数回调和异常处理。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 文件使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/17-json/。

1. [parse-and-format.mjs](scripts/17-json/parse-and-format.mjs)：文本解析、序列化和缩进。
2. [invalid-json-error.mjs](scripts/17-json/invalid-json-error.mjs)：尾随逗号导致解析失败。
3. [missing-values.mjs](scripts/17-json/missing-values.mjs)：被省略的对象字段和数组中的 null。
4. [replacer.mjs](scripts/17-json/replacer.mjs)：白名单与函数式转换。
5. [reviver.mjs](scripts/17-json/reviver.mjs)：后序处理、删除字段与恢复大整数。
6. [custom-representation.mjs](scripts/17-json/custom-representation.mjs)：自定义表示、Date 与大整数编码。
7. [bigint-error.mjs](scripts/17-json/bigint-error.mjs)：默认 BigInt 序列化失败。
8. [precision-and-copy.mjs](scripts/17-json/precision-and-copy.mjs)：数值精度与共享身份丢失。
9. [cycle-error.mjs](scripts/17-json/cycle-error.mjs)：循环引用序列化失败。
10. [validate.mjs](scripts/17-json/validate.mjs)：解析之后执行字段校验。

## 1 JSON 文本与 JavaScript 值

JSON 是数据交换文本格式，值可为对象、数组、字符串、数字、布尔值或 null；对象键与字符串都用双引号，不支持注释、尾随逗号、undefined、BigInt 字面量或函数。JSON.parse 把文本转成值，JSON.stringify 把受支持的值转成文本。

stringify 默认访问对象可枚举的自有字符串键；Symbol 键不进入结果。第三参数 space 可用非负数字表示缩进空格数，上限为 10；用字符串时最多取前 10 个 UTF-16 码元。为生成标准 JSON，缩进字符串应使用 JSON 允许的空白字符，而不是任意装饰文字。

[parse-and-format.mjs](scripts/17-json/parse-and-format.mjs)：

```javascript
const text = '{"title":"JS","minutes":30,"done":false}';
const lesson = JSON.parse(text);
console.log(lesson.title, lesson.minutes, lesson.done);
const formatted = JSON.stringify(lesson, null, 2);
console.log(formatted);
console.log(JSON.parse(formatted).minutes === lesson.minutes);
console.log(JSON.parse("null") === null, JSON.parse("12"));

// 按本例输入运行，输出依次为：
// JS 30 false
// {
//   "title": "JS",
//   "minutes": 30,
//   "done": false
// }
// true
// true 12
```

Step 1：运行本节示例。

```bash
node scripts/17-json/parse-and-format.mjs
```

[invalid-json-error.mjs](scripts/17-json/invalid-json-error.mjs)：

```javascript
JSON.parse('{"minutes":30,}');

// 独立运行：退出状态为 1；诊断包含 SyntaxError；Expected double-quoted property name。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/17-json/invalid-json-error.mjs
```

## 2 缺失值、null 与不支持的值

JSON 没有 undefined。对象属性的值为 undefined、函数或 Symbol 时，默认序列化会省略该属性；数组中对应位置以及空洞会变为 null，以保留位置。顶层 undefined、函数或 Symbol 会使 stringify 返回 undefined，而不是字符串。

NaN 和正负 Infinity 序列化为 null；负零序列化后不能保持与正零的区别。null 则是明确存在的空值，因此“没有字段”和“字段值是 null”需要应用自行区分。默认序列化 Map/Set 不会遍历它们的内部条目，必须先转成约定的数据结构。

[missing-values.mjs](scripts/17-json/missing-values.mjs)：

```javascript
const key = Symbol("hidden");
const input = { absent: undefined, empty: null, callback() {}, token: Symbol("v"), [key]: 1 };
const text = JSON.stringify(input);
console.log(text);
const restored = JSON.parse(text);
console.log(Object.hasOwn(restored, "absent"), Object.hasOwn(restored, "empty"));
console.log(JSON.stringify([undefined, () => 1, Symbol("v"), , NaN, Infinity]));
console.log(JSON.stringify(undefined) === undefined, JSON.stringify(-0));
console.log(JSON.stringify(new Map([["JS", 1]])), JSON.stringify(new Set([1])));
console.log(JSON.stringify([...new Map([["JS", 1]])]));

// 按本例输入运行，输出依次为：
// {"empty":null}
// false true
// [null,null,null,null,null,null]
// true 0
// {} {}
// [["JS",1]]
```

Step 1：运行本节示例。

```bash
node scripts/17-json/missing-values.mjs
```

## 3 replacer 筛选和变换

stringify 的第二参数可以是函数，也可以是包含字符串或数字的数组。数组形式指定对象属性的包含名单，且会作用于嵌套对象；它不是仅针对顶层的投影操作。函数形式接收 key 与 value，返回要继续序列化的值；返回 undefined 会删除对象属性，但对数组元素仍生成 null。

函数先对包含整个输入的临时容器调用一次，根 key 为空字符串，之后再递归处理属性。输入对象也可能真的有空字符串键，不能不加条件地把所有空 key 都当成根。

[replacer.mjs](scripts/17-json/replacer.mjs)：

```javascript
const data = { title: "JS", minutes: 30, internal: "草稿", detail: { minutes: 5, internal: "临时" } };
console.log(JSON.stringify(data, ["title", "minutes", "detail"]));
const transformed = JSON.stringify(data, (key, value) => {
  if (key === "internal") return undefined;
  if (key === "minutes") return value * 60;
  return value;
});
console.log(transformed);
console.log(JSON.stringify([1, 2], (key, value) => key === "0" ? undefined : value));

// 按本例输入运行，输出依次为：
// {"title":"JS","minutes":30,"detail":{"minutes":5}}
// {"title":"JS","minutes":1800,"detail":{"minutes":300}}
// [null,2]
```

Step 1：运行本节示例。

```bash
node scripts/17-json/replacer.mjs
```

## 4 reviver 恢复约定类型

parse 的 reviver 在子属性处理完后处理父属性，最终处理根。返回值替换当前属性，返回 undefined 则删除属性；处理根时返回 undefined 会让整个 parse 返回 undefined。数组元素被删除后会留下空洞，不会像 stringify 那样自动变成 null。

恢复日期或大整数前先设计明确字段约定，不应把任意外形相似的字符串都改成另一种类型。这里仅把 id 字段的十进制字符串恢复为 BigInt；第三参数 source 文本访问不属于本章 ECMAScript 2025 基线，不能把它当作所有 2025 实现都支持的解析能力。

[reviver.mjs](scripts/17-json/reviver.mjs)：

```javascript
const visited = [];
const value = JSON.parse('{"id":"9007199254740993","nested":{"drop":1,"keep":2}}', (key, item) => {
  visited.push(key === "" ? "<root>" : key);
  if (key === "drop") return undefined;
  if (key === "id" && typeof item === "string" && /^[0-9]+$/.test(item)) return BigInt(item);
  return item;
});
console.log(value.id === 9007199254740993n, Object.hasOwn(value.nested, "drop"));
console.log(visited.join(","));
const sparse = JSON.parse("[1,2]", (key, item) => key === "0" ? undefined : item);
console.log(sparse.length, 0 in sparse, sparse[1]);
console.log(JSON.parse("1", () => undefined) === undefined);

// 按本例输入运行，输出依次为：
// true false
// id,drop,keep,nested,<root>
// 2 false 2
// true
```

Step 1：运行本节示例。

```bash
node scripts/17-json/reviver.mjs
```

## 5 toJSON、日期与 BigInt

遇到具有可调用 toJSON 的对象时，stringify 先调用它，然后把结果交给 replacer。toJSON 决定数据表示，并非保证原对象结构都保留。Date 的 toJSON 对有效时间返回 UTC ISO 字符串，JSON.parse 默认不会把它恢复为 Date；无效 Date 的 toJSON 返回 null。

BigInt 默认不能序列化，会抛 TypeError。可在应用边界显式编码为十进制字符串，并在约定字段上恢复；不要修改 BigInt.prototype 来改变整个进程的全局行为。时间与时区的完整规则在下一章展开。

[custom-representation.mjs](scripts/17-json/custom-representation.mjs)：

```javascript
const stages = [];
const item = {
  title: "JS",
  internal: "草稿",
  toJSON(key) { stages.push("toJSON:" + key); return { title: this.title }; }
};
console.log(JSON.stringify({ item }, (key, value) => {
  if (key === "item") stages.push("replacer:item");
  return value;
}));
console.log(stages.join(","));
const text = JSON.stringify({ createdAt: new Date("2025-01-02T03:04:05Z") });
console.log(text, typeof JSON.parse(text).createdAt);
console.log(JSON.stringify(new Date(NaN)));
console.log(JSON.stringify({ id: 9007199254740993n }, (key, value) =>
  typeof value === "bigint" ? value.toString() : value));

// 按本例输入运行，输出依次为：
// {"item":{"title":"JS"}}
// toJSON:item,replacer:item
// {"createdAt":"2025-01-02T03:04:05.000Z"} string
// null
// {"id":"9007199254740993"}
```

Step 1：运行本节示例。

```bash
node scripts/17-json/custom-representation.mjs
```

[bigint-error.mjs](scripts/17-json/bigint-error.mjs)：

```javascript
JSON.stringify({ id: 1n });

// 独立运行：退出状态为 1；诊断包含 TypeError；Do not know how to serialize a BigInt。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/17-json/bigint-error.mjs
```

## 6 精度、循环引用与复制边界

JSON 的数字文本转换成 JavaScript Number 时受 Number 精度限制，超过安全整数范围的编号可能在 reviver 收到它以前就已舍入；事后再 BigInt(number) 不能恢复丢失的数字。需要精确编号时，从发送端就编码为字符串。

JSON 只能表示树形数据，不能表达对象身份。非循环共享引用可以序列化，但重新 parse 后的两个位置成为两个对象；真正的循环引用则让 stringify 抛 TypeError。原型、方法、属性描述符及自定义类型也不会被通用恢复，所以 stringify 后 parse 不是任意对象的深复制算法。

[precision-and-copy.mjs](scripts/17-json/precision-and-copy.mjs)：

```javascript
const rounded = JSON.parse('{"id":9007199254740993}');
console.log(rounded.id, Number.isSafeInteger(rounded.id));
const exact = JSON.parse('{"id":"9007199254740993"}');
console.log(BigInt(exact.id) === 9007199254740993n);
const shared = { minutes: 5 };
const original = { first: shared, second: shared };
const copy = JSON.parse(JSON.stringify(original));
console.log(original.first === original.second, copy.first === copy.second);
copy.first.minutes = 8;
console.log(copy.second.minutes, original.first.minutes);

// 按本例输入运行，输出依次为：
// 9007199254740992 false
// true
// true false
// 5 5
```

Step 1：运行本节示例。

```bash
node scripts/17-json/precision-and-copy.mjs
```

[cycle-error.mjs](scripts/17-json/cycle-error.mjs)：

```javascript
const record = { title: "JS" };
record.self = record;
JSON.stringify(record);

// 独立运行：退出状态为 1；诊断包含 TypeError；Converting circular structure to JSON。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/17-json/cycle-error.mjs
```

## 7 解析成功不等于业务有效

JSON.parse 只验证 JSON 文法，不验证应用期待的字段、类型、范围和额外键。输入 null、数组或字段值为负数都可能是有效 JSON，却不符合“学习记录”这个业务结构。解析后应建立单独校验边界，再交给计算逻辑。

下例只接受具有 title 和 minutes 两个自有字段的对象，title 必须非空，minutes 必须是非负安全整数；这是本例的明确约定。不要用 eval 解析 JSON，也不要把外部任意键直接合并到带原型的配置对象。

[validate.mjs](scripts/17-json/validate.mjs)：

```javascript
function parseLesson(text) {
  // 先让 JSON.parse 检查语法，再按本节约定检查对象和字段。
  const value = JSON.parse(text);
  if (value === null || typeof value !== "object" || Array.isArray(value)) {
    throw new TypeError("学习记录必须是对象");
  }
  if (!Object.hasOwn(value, "title") || !Object.hasOwn(value, "minutes") ||
      Object.keys(value).some(key => key !== "title" && key !== "minutes") ||
      typeof value.title !== "string" || value.title.trim() === "" ||
      !Number.isSafeInteger(value.minutes) || value.minutes < 0) {
    throw new TypeError("学习记录字段不符合约定");
  }
  return { title: value.title, minutes: value.minutes };
}
console.log(JSON.stringify(parseLesson('{"title":"JS","minutes":30}')));
// 这两个反例专门展示校验边界；只捕获预期类型，其他异常直接传播。
for (const input of ['null', '{"title":"JS","minutes":-1}']) {
  try { parseLesson(input); }
  catch (error) {
    if (!(error instanceof TypeError)) throw error;
    console.log(error.message);
  }
}

// 按本例输入运行，输出依次为：
// {"title":"JS","minutes":30}
// 学习记录必须是对象
// 学习记录字段不符合约定
```

Step 1：运行本节示例。

```bash
node scripts/17-json/validate.mjs
```

## 本章小结

- JSON 文本有独立文法，缺失字段、null 和数组空位的转换不同。
- replacer、reviver、toJSON 是明确的数据边界，不会自动恢复所有类型。
- 精度和对象身份可能丢失；解析后的业务结构仍需校验。

## 练习

1. 定义只允许 title 与 completed 两个字段的数据协议并校验。可核对标准：completed 为 boolean，额外键、缺少键与 null 根值都被拒绝。
2. 序列化一个带 BigInt 编号和 Date 时间的对象，再按字段约定恢复。可核对标准：编号逐位相同，日期恢复后的 getTime 与原值相同。
3. 比较共享子对象和循环引用的 JSON 行为。可核对标准：前者能转成文本但身份丢失，后者在 stringify 时抛 TypeError；说明两者差别。

## 参考与引用来源

- TC39 官方 ECMAScript 2025 分页版：[§25.5 JSON 文法及 parse/stringify](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-json-object)；[§25.5.1.2 reviver 次序与删除](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-internalizejsonproperty)；[§25.5.2.2 toJSON、replacer 与类型转换](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-serializejsonproperty)；[§25.5.2.4–5 循环检查与数组转换](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-serializejsonobject)；[§21.4.4.37 Date.toJSON](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-date.prototype.tojson)。
- MDN 用法对照：[语法、reviver 和精度边界](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/JSON/parse)；[忽略值、循环引用和对象复制边界](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/JSON/stringify)。